In [ ]:
import re
import json
import time
import requests


def extract_prompt(image_id):
    html = requests.get(f"https://civitai.com/images/{image_id}", timeout=20).text

    start = html.find('"prompt":"')

    if start == -1:
        return None

    start += len('"prompt":"')

    end = html.find('","cfgScale"', start)

    if end == -1:
        return None

    raw_prompt = html[start:end]

    try:
        return json.loads(f'"{raw_prompt}"')
    except json.JSONDecodeError:
        return None

In [ ]:
# Download IDs from CivitAI posts.
i = 0
image_ids = []
url = "https://civitai.com/api/v1/images"
params = {"limit": 100}

while True:
    try:
        data = requests.get(url, params=params).json()
        image_ids.extend(item["id"] for item in data["items"])
    except Exception as e:
        print(f'Error: {e}')
        time.sleep(15)
        continue

    print(f"Total collected: {len(image_ids)} - page {i+1}")

    next_page = data.get("metadata", {}).get("nextPage")

    if not next_page:
        break

    url = next_page
    params = {}
    time.sleep(1)
    i += 1

print(f"Total final: {len(image_ids)}")

In [ ]:
# Download prompts.
prompts = {}

for i, image_id in enumerate(image_ids):
    if image_id in prompts:
        continue
        
    try:
        time.sleep(0.2)
        prompt = extract_prompt(image_id)

        if prompt is not None and 'civitai' not in str(prompt).lower():
            prompts[image_id] = prompt
        else:
            print(f'Prompt associated to ID {image_id} is None')
    except Exception as e:
        print(f'Error: {e}')
        time.sleep(15)
        continue

    if i % 100 == 0:
        print(f'{i+1} de {len(image_ids)} (len  = {len(prompts)})')
        
        with open("prompts.json", "w", encoding="utf-8") as f:
            json.dump(prompts, f, ensure_ascii=False, indent=4)

print(len(prompts))
prompts = list(set(prompts))
print(len(prompts))

In [ ]:
with open("prompts.json", "w", encoding="utf-8") as f:
    json.dump(prompts, f, ensure_ascii=False, indent=4)